# 🚀 Automatische Erkennung von Einstellungen mit *TransNetV2*

Dieses Notebook annotiert die Abfolge und die Länge der Einstellungen innerhalb einer oder mehrerer Videodateien mithilfe der **Shot Boundary Detection**, also der automatisierten Erkennung von Übergängen zwischen Einstellungen. 

Der Arbeitsablauf ist in vier Abschnitte gegliedert: **Festlegen der Parameter**, **technische Vorbereitung**, **Analyse der Einstellungsübergänge** und **Export der Ergebnisse**. Die Texte zwischen den Codeblöcken erläutern vor allem den Zweck dieser einzelnen Schritte und die Bedeutung der erzeugten Dateien, während die Kommentare innerhalb des Codes auf Englisch dessen Funktionsweise vermitteln.

Weitere Informationen zur Funktionsweise dieses Notebooks bietet das <a href="https://quadriga-dk.github.io/Bewegtes-Bild-Fallstudie-4/annotation/einstellung_intro.html" class="external-link" target="_blank">Einführungskapitel zur Einstellungserkennung innerhalb der OER</a>.

## Ausführung des Notebooks

In der Jupyter-Book-Version der OER sind die Ergebnisse der Ausführung bereits gespeichert. Sie können daher betrachtet werden, ohne den Code selbst auszuführen. ***Für ein tieferes Verständnis der Lerninhalte und der Funktionen des Codes und für die Bearbeitung der späteren Übungen empfehlen wir jedoch, das Notebook in einer Cloud-Umgebung oder lokal auszuführen***.

**Cloud-Umgebung:** Klicken Sie oben rechts in der Menüleiste auf das Raketen-Symbol 🚀 und wählen Sie „Colab“, um das Notebook auszuführen. Wir empfehlen für die Ausführung eine GPU zu verwenden. Lesen sie hierzu die Hinweise im Kapitel zu <a href="https://quadriga-dk.github.io/Bewegtes-Bild-Fallstudie-4/annotation/technik.html#ausfuhrungsmodi" class="external-link" target="_blank">Ausführungs-Modi</a>. Falls eine GPU nicht zur Verfügung stehen, sollte ist die Nutzung einer CPU ebenfalls möglich, die Verarbeitungszeiten werden sich dadurch jedoch verlängern.

**Lokale Umgebung:** Klicken Sie oben rechts in der Menüleiste auf das Download-Symbol ↓ und wählen Sie „.ipynb“, um das Notebook herunterzuladen und lokal auszuführen. Abhängig von Browser und Betriebssystem müssen Sie die Datei gegebenenfalls per Rechtsklick herunterladen.

***Wir empfehlen außerdem, die Codeblöcke einzeln und nacheinander auszuführen, statt Funktionen wie „Alle ausführen“ in Colab oder „Run All Cells“ in anderen Anwendungen zu verwenden***. Die separate Ausführung der Codeblöcke erleichtert das Verständnis der damit verbundenen Funktionen und stellt sicher, dass alle vorbereitenden Schritte vor den nachfolgenden Codeblöcken ausgeführt wurden.


## Festlegen der Parameter

In diesem Abschnitt werden die zentralen Parameter für die spätere Analyse festgelegt. Sie können an dieser Stelle angepasst werden, ohne Änderungen im weiteren Code vornehmen zu müssen. Zum Erlernen der Funktionen des Codes in diesem Notebook und für einen ersten Probelauf empfiehlt es sich, die voreingestellten Werte beizubehalten.

Die Variable `VIDEO_PATH` benennt den Ordner, innerhalb dessen die zu analysierenden Videodateien abgelegt werden. Falls dieser noch nicht existieren sollte, wird ein Ordner mit entsprechender Benennung angelegt. Die einzelnen Videos können dabei sowohl in Unterordnern als auch auf der höchsten Ebene des Ordners abgelegt werden. **Wichtig** ist, dass jede Videodatei einen einzigartigen Namen tragen muss. 

In diesen Ordner werden auch die für diese *OER* vorbereiteten Testdateien im `.zip`-Format abgelegt. Diese sind für die Ausführung dieses Jupyter Notebooks zu Lernzwecken vorgesehen und wir empfehlen, diese hierfür auch zu verwenden. Ist deren Download nicht erwünscht, kann die Variable `DOWNLOAD_TESTFILES` auf `False` gesetzt werden. In diesem Fall müssen eigene Videodateien in den Ordner kopiert werden, der unter dem Parameter `VIDEO_PATH` angegeben ist.

Zusätzlich können auch eigene `.zip`-Archive von Online-Speicherplätzen heruntergeladen werden. Hierzu muss ein entsprechender Download-Link innerhalb von Anführungszeichen dem Parameter `CUSTOM_ZIP_LINK` zugeordnet werden, z.B. `CUSTOM_ZIP_LINK = "https://example.org/video_archive.zip"`. Wichtig ist dabei, dass es sich um einen Direktlink zur `.zip`-Datei handelt, nicht um einen Link zu einer Website. Die Option zum Download eigener Dateien ist besonders bei der Analyse von Langfilmen in *Google Colab* hilfreich, da die Videodateien dadurch nicht manuell in die *Colab*-Umgebung hochgeladen werden müssen.

Der Parameter `EXPORT_PATH` bezeichnet den Pfad zum Exportordner. Auch dieser wird automatisch angelegt.

### Festlegung des Schwellenwerts

Besonders wichtig ist der Schwellenwert `THRESHOLD`. Er bestimmt, ab welcher vom Modell berechneten Wahrscheinlichkeit ein Bildwechsel als Einstellungsgrenze gewertet und als Annotation in die Ausgabe der Ergebnisse aufgenommen wird. Ein höherer Wert führt tendenziell zu weniger, aber sichereren Treffern. Ein niedrigerer Wert führt zur Erkennung von mehr Übergängen, erhöht allerdings auch zu einer höheren Wahrscheinlichkeit falsch-positiver Ergebnisse ("*false positives*"). Der Werteumfang für die Wahrscheinlichkeit liegt dabei zwischen `0.0` für 0% und `1.0` für 100%. Eine ausführlichere Einführung zu Schwellenwerten finden Sie im <a href="https://quadriga-dk.github.io/Bewegtes-Bild-Fallstudie-4/vision/begriffe.html#schwellenwert" class="external-link" target="_blank">OER-Abschnitt zu zentralen Begriffen und Konzepten</a>.



In [1]:
import os

# Path to the directory containing the video files to be analysed.
# The default relative path is resolved from the location where the notebook is executed.
VIDEO_PATH = "videos_shots"

# The video directory is created automatically if it does not exist yet.
# `exist_ok=True` prevents an error if the directory already exists.
os.makedirs(VIDEO_PATH, exist_ok=True)

# Path to the directory for the export of the analysis results
EXPORT_PATH = "export_shots"
os.makedirs(EXPORT_PATH, exist_ok=True)

# If True, a provided test video is downloaded and extracted.
# Set this option to False when analyzing your own files.
DOWNLOAD_TESTFILES = True

# Add your own custom link inside quotes to download additional files into the video folder.
CUSTOM_ZIP_LINK = "insert link here"
# Change the name of the custom archives. A new archive with the same name will not be downloaded. 
CUSTOM_ARCHIVE_NAME = "custom_archive"

# Minimum probability at which TransNetV2 treats a frame as a confirmed shot boundary.
# The value ranges from 0 to 1.
THRESHOLD = 0.5

# Separator line for visually clearer text output.
DIVIDER = "─" * 100

---
## Technische Vorbereitung

Bevor ein Video analysiert werden kann, müssen die benötigten Softwarepakete installiert, in das Notebook eingebunden und das vortrainierte TransNetV2-Modell geladen werden.

### Installation der Programmbibliotheken

Der folgende Codeblock installiert externe Programmbibliotheken, die nicht in der Standardinstallation von *Python* enthalten sind. Dazu zählen neben dem Modell `TransNetV2` zur Erkennung der Einstellungsübergänge
- `pandas` zum Bearbeiten und Exportieren von tabellarischen Daten,
- `opencv-python` und `ffmpeg` zur Videobearbeitung,
- und `pympi-ling` zum Schreiben von *ELAN*-Dateien (`.eaf`).

Für die wichtigsten Bibliotheken wird eine spezifische Version installiert, um deren interdependente Funktionsweise zu garantieren.

Zunächst wird geprüft, ob eine *CUDA*-fähige NVIDIA-GPU vorhanden ist und die Installation entsprechend angepasst. Weitere Informationen zu CPU und GPU finden Sie im <a href="https://quadriga-dk.github.io/Bewegtes-Bild-Fallstudie-4/annotation/technik.html#verwendete-hardware" class="external-link" target="_blank">OER-Abschnitt zu den technischen Grundlagen</a>.

Bei einer erneuten lokalen Ausführung des Notebooks in identischer Arbeitsumgebung muss diese Zelle nicht bei jeder Sitzung erneut ausgeführt werden. In *Google Colab* ist eine erneute Installation nach dem Neustart der Laufzeit hingegen notwendig, da alle speziell installierten Programmbibliotheken beim Beenden der Laufzeit gelöscht werden.



In [2]:
print("Installing dependencies.")
import importlib.util
import os
import sys

# Check for an NVIDIA GPU.
if sys.platform.startswith("win"):
    cuda = os.system("nvidia-smi > NUL 2>&1") == 0
else:
    cuda = os.system("nvidia-smi > /dev/null 2>&1") == 0

# Select a compatible PyTorch wheel only if PyTorch is not already installed.
if importlib.util.find_spec("torch") is None:
    if cuda:
        !pip install -q torch --index-url https://download.pytorch.org/whl/cu126
    elif sys.platform == "darwin":
        !pip install -q torch
    else:
        !pip install -q torch --index-url https://download.pytorch.org/whl/cpu

# Install the model, video-processing, annotation and progress visualisation dependencies.
!pip install -q transnetv2-pytorch==1.0.5
!pip install -q pandas opencv-python==5.0.0.93 
!pip install -q static-ffmpeg==3.0 
!pip install -q pympi-ling
!pip install -q tqdm==4.68.3

# Replace a CPU-only PyTorch build if an NVIDIA GPU is available.
import torch
if cuda and not torch.cuda.is_available():
    print("NVIDIA GPU detected, but PyTorch cannot use CUDA. Installing CUDA-enabled PyTorch.")
    !pip install -q --upgrade --force-reinstall torch --index-url https://download.pytorch.org/whl/cu126
    raise RuntimeError("Restart the kernel after replacing PyTorch, then run the notebook again.")

print("Dependencies installed.")


Installing dependencies.
Dependencies installed.


### Import der Programmbibliotheken

Durch den Import werden die installierten Funktionen nun in der aktuellen Notebook-Sitzung verfügbar. Die Bibliotheken übernehmen unterschiedliche Aufgaben, die im Codeblock jeweils kurz erläutert werden.


In [3]:
# Switch the progress bar renderer.
#from tqdm.notebook import tqdm as notebook_tqdm
# Pretrained neural network for shot-boundary detection.
from transnetv2_pytorch import TransNetV2

#sys.modules[TransNetV2.__module__].tqdm = notebook_tqdm
# Check which device PyTorch can use.
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Handling file and directory paths.
from pathlib import Path

# Editing and exporting tabular data.
import pandas as pd

# Writing ELAN files.
import pympi

# Representing and calculating time values.
import datetime
import time

# Reading videos and their technical properties.
import cv2
import static_ffmpeg

# Downloading a file from a URL and extracting ZIP archives.
from urllib.request import urlretrieve
import zipfile

print("Dependencies imported.")


PyTorch: 2.14.0
CUDA available: False
Dependencies imported.


### Download des Analysematerials

Falls bei der [Festlegung der Parameter](#festlegen-der-parameter) `DOWNLOAD_TESTFILES = True` gesetzt ist, lädt das Notebook das `.zip`-Archiv `videos_shots.zip` herunter, das den Trailer zu Konrad Wolfs Film *Der geteilte Himmel* enthält. Anschließend wird ihr Inhalt in den zuvor festgelegten Videoordner `VIDEO_PATH` entpackt. Zu Lernzwecken im Rahmen der OER empfehlen wir den Download und die Arbeit mit den Testdateien im Notebook. Herkunft und Zusammensetzung des Testarchivs erläutert der OER-Abschnitt <a href="https://quadriga-dk.github.io/Bewegtes-Bild-Fallstudie-4/annotation/einstellung_intro.html#analysematerial" class="external-link" target="_blank">zum Analysematerial</a>.

Dieser Schritt ist für die Arbeit mit eigenen Video- und Film-Korpora nicht erforderlich. Eigene Videodateien können direkt im `VIDEO_PATH` Ordner oder in dessen Unterordnern abgelegt werden. In diesem Fall kann `DOWNLOAD_TESTFILES = False` gesetzt werden. Außerdem kann mittels des Parameters `CUSTOM_ZIP_LINK` ein eigenes `.zip`-Archiv heruntergeladen werden, wie bereits eingangs im Kontext der [Festlegung der Parameter](#festlegen-der-parameter) erläutert. In beiden Fällen sollte der Codeblock zur [Festlegung der Parameter](#festlegen-der-parameter) bereits ausgeführt worden sein, um, falls notwendig den Videoordner erst zu erstellen.

In [4]:
# Define a reusable function for downloading .zip archives.
# This function takes three arguments: The download link, the name for the downloaded archive, and the directory to unzip the archive into.
def download_zip(url: str, archive_name: str, target_directory: str):

    # Add .zip to the filename and turn it into a relative path in the working directory.
    archive_path = Path(f"{archive_name}.zip")

    # Download only if the archive is not already present in the working directory.
    if not archive_path.is_file():
        # Download the file.
        print(f"Downloading {archive_path}...")
        urlretrieve(url, archive_path)
        print(f"{archive_name} downloaded.")
    else:
        print(f"{archive_name} already downloaded.")

    # Unzip the file to the target_directory.
    target_directory = Path(target_directory)
    with zipfile.ZipFile(archive_path,"r") as zip_ref:
        zip_ref.extractall(target_directory)
        print(f"Extracted {archive_name} to {target_directory}.")

# Download and extract the test files if the setting is enabled. 
if DOWNLOAD_TESTFILES == True:
    print("Testfile download requested.")
    
    # Download the testfiles
    url = ("https://owncloud.gwdg.de/index.php/s/Mw2EDXpprzJVn2v/download?path=%2F&files=videos_shots.zip")
    download_zip(url, "videos_shots", VIDEO_PATH)
else:
     print("No testfile download requested.")

# Download and extract the custom archive if a link is provided.
if CUSTOM_ZIP_LINK != "insert link here":
    print("Custom file download requested.")
 
    # Download the custom video archive.
    download_zip(CUSTOM_ZIP_LINK, CUSTOM_ARCHIVE_NAME, VIDEO_PATH)

Testfile download requested.
videos_shots already downloaded.
Extracted videos_shots to videos_shots.


### Vorbereitung des `TransNetV2`-Modells

In diesem Schritt wird das *Machine-Learning-Modell* <a href="https://quadriga-dk.github.io/Bewegtes-Bild-Fallstudie-4/annotation/einstellung_intro.html#funktionsweise-des-transnetv2-modells" class="external-link" target="_blank">TransNetV2</a> zur Erkennung der Einstellungsübergänge vorbereitet. Es handelt sich um ein bereits trainiertes neuronales Netz. Es berechnet für jeden Frame eines Videos einen Wert zwischen `0.0` und `1.0`, der angibt, wie wahrscheinlich es ist, dass an dieser Stelle ein Übergang zwischen zwei Einstellungen stattfindet.


In [5]:
# Select CUDA automatically when it works, otherwise use the CPU.
model = TransNetV2(device="auto")

# Disable training-specific behaviour for reliable inference.
model.eval()
print(f"Model loaded on: {next(model.parameters()).device}")

# Make the bundled ffmpeg and ffprobe executables available for video decoding. 
static_ffmpeg.add_paths()


ℹ️  MPS device detected but not used due to numerical inconsistency issues.
   Use --device mps to explicitly enable MPS (faster but inconsistent results).
Model loaded on: cpu


True

---
## Analyse der Einstellungsübergänge

Hier beginnt die eigentliche Analyse. Zunächst werden die abgelegten Videodateien erfasst und deren Metadaten gesammelt. Anschließend führt das Notebook mithilfe des Modells die eigentliche Einstellungserkennung durch und bereitet deren Ergebnis tabellarisch auf.

### Erfassung der technischen Videometadaten

Für jede Videodatei werden technische Metadaten wie Auflösung und Framerate gespeichert.
Diese Angaben sind sowohl für die Interpretation des Analyseergebnisses als auch für die spätere Umrechnung von Framepositionen in Sekunden notwendig.

In [6]:
# Supported video file extensions.
VIDEO_EXTENSIONS = {
    ".mp4", ".avi", ".mov", ".mkv", ".wmv",
    ".flv", ".webm", ".mpeg", ".mpg", ".m4v", ".3gp"
}

# Store each video's path and technical metadata.
# Video filenames must be unique across all subdirectories.
videos = {}

# Search the input directory for video files.
for video_path in Path(VIDEO_PATH).rglob("*"):

    # Ignore directories and other non-file entries.
    if not video_path.is_file():
        continue

    # Ignore files inside Jupyter checkpoint directories.
    if ".ipynb_checkpoints" in video_path.parts:
        continue

    # Ignore unsupported file formats.
    if video_path.suffix.lower() not in VIDEO_EXTENSIONS:
        continue

    # Use the filename without its extension as the dictionary key.
    video_name = video_path.stem
    # Extract the release year from the filename and check if it is an integer
    try:
        video_title, release_year = video_name.rsplit("_", 1)
        release_year = int(release_year)
        print(f"Successfully extracted video name {video_title} and release year {release_year}.")
    except:
        release_year = None
        video_title = video_name
        print(f"Filename {video_name} does not provide a release year. The correct formatting is 'title_year'.")

    if video_name in videos:
        raise ValueError(
            f"Duplicate video name: {video_name}. "
            "Please use unique filenames."
        )

    # Open the video without loading the complete file into memory.
    capture = cv2.VideoCapture(str(video_path))

    if not capture.isOpened():
        print(f"Video could not be opened: {video_path}")
        capture.release()
        continue

    # Read the video's technical metadata.
    frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_rate = capture.get(cv2.CAP_PROP_FPS)

    # Release the video file after reading its metadata.
    capture.release()

    # Skip files with invalid metadata.
    if frame_rate <= 0 or frame_count <= 0:
        print(f"Video contains invalid metadata: {video_path}")
        continue

    # Calculate the video duration and convert it to HH:MM:SS.
    duration_hh_mm_ss = str(
        datetime.timedelta(seconds=round(frame_count / frame_rate))
    )

    print(f"Video file: {video_name}")
    print(
        f"Resolution: {frame_width}x{frame_height} | "
        f"Frames: {frame_count} | "
        f"Frame Rate: {frame_rate:.3f} | "
        f"Duration: {duration_hh_mm_ss}"
    )
    print(DIVIDER)

    # Initialise the video dictionary.
    videos[video_name] = {
        "path"              : video_path,
    }
    # Store the metadata for the subsequent analysis steps.
    videos[video_name]["metadata"]= {
        "video_name"        : video_path.name,
        "title"             : video_title,
        "release_year"      : release_year,
        "frame_count"       : frame_count,
        "frame_height"      : frame_height,
        "frame_width"       : frame_width,
        "frame_rate"        : frame_rate,
        "duration_hh_mm_ss"       : duration_hh_mm_ss,
        "model"                   : "transnetv2",
        "device"                  : next(model.parameters()).device.type,
        "analysis_timestamp"      : None,  # Updated when analysis begins.
        "processing_time_hh_mm_ss": None   # Updated after the analysis.
    }

Successfully extracted video name Der geteilte Himmel_Trailer and release year 1964.
Video file: Der geteilte Himmel_Trailer_1964
Resolution: 1920x1080 | Frames: 5225 | Frame Rate: 25.000 | Duration: 0:03:29
────────────────────────────────────────────────────────────────────────────────────────────────────


### Anwendung des Modells

Im folgenden Codeblock werden nacheinander die zuvor gesammelten Videos analysiert. 

Für jedes Video berechnet `TransNetV2` pro Frame die Wahrscheinlichkeit für einen Einstellungsübergang. Diese Wahrscheinlichkeiten werden anschließend mithilfe des gewählten Schwellenwerts `THRESHOLD` in konkrete Einstellungssegmente (`shots`) übertragen. `THRESHOLD` wird im Codeblock zur [Festlegung der Parameter](#festlegen-der-parameter) eingestellt.

Dieser Schritt ist sehr rechenintensiv! Je nach Videomaterial kann er von einer halben Stunde bis zu mehreren Stunden in Anspruch nehmen. 

In [ ]:
# Destination directory for later export files.
export_directory = Path(EXPORT_PATH)

# Analyze each previously registered video in sequence.
for video_name, video in videos.items():
   # Record the start of this analysis in UTC and start the duration timer.
   video["metadata"]["analysis_timestamp"] = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
   start_time = time.perf_counter()

   # Calculate predictions without calculating gradients.
   # Gradients are only required when training models.
   # This reduces memory usage and speeds up computation.
   with torch.no_grad():
      # The predict_video() functions returns three values: the extracted frames, single frame predictions and all frame predictions.
      # For the detection of shot boundaries within this notebook, only the single frame predictions are needed.
      # For more information, refer to the TransNetV2 paper.
      _, single_frame_predictions, _ = model.predict_video(video["path"])

      # Convert the PyTorch result to a NumPy array for further processing.
      prediction_scores = single_frame_predictions.cpu().detach().numpy()

      # Convert the probability scores into concrete shot segments.
      shots = model.predictions_to_scenes(prediction_scores, THRESHOLD)

      # Calculate the elapsed time from when the timer was started.
      elapsed_time_seconds = time.perf_counter() - start_time
      # Convert the elapsed time into the hhmmss format
      elapsed_time_hh_mm_ss = str(
         datetime.timedelta(seconds=round(elapsed_time_seconds))
         )

      print(f"Shot Boundary Detection on {video_name} took {elapsed_time_hh_mm_ss} of processing time.")
      print(DIVIDER)

      # Record the compute device used by the model.
      video["metadata"]["device"] = next(model.parameters()).device.type

      # Add the analysis result to the video dictionary
      video.update({
         "shots" : shots,
         "elapsed_time_hh_mm_ss" : elapsed_time_hh_mm_ss,
         "prediction_scores": prediction_scores
      })

Extracting frames from videos_shots/Der geteilte Himmel_Trailer_1964.mp4


Processing frames: 100%|██████████| 5225/5225 [01:02<00:00, 83.32frame/s]

Shot Boundary Detection on Der geteilte Himmel_Trailer_1964 took 0:01:07 of processing time.
────────────────────────────────────────────────────────────────────────────────────────────────────

### Tabellarische Aufbereitung des Ergebnisses

Die vom Modell gelieferten Einstellungen (`shots`) werden in eine Tabelle (`df_shots`) überführt, mittels dessen sich Datensätze einfacher bearbeiten und exportieren lassen. 

Neben Startframe (`start`) und Endframe (`end`) werden der zeitliche Mittelpunkt (`middle`) und die Dauer (`duration`) jeder Einstellung berechnet. Anschließend werden alle Frameangaben mithilfe der Bildrate (`frame_rate`) in Sekunden umgerechnet und auf zwei Nachkommastellen gerundet. Die Sekundenwerte erleichtern die weitere Auswertung der erkannten Einstellungen.


In [8]:
# Process the analysis results for each previously registered video in sequence.
for video_name, video in videos.items():
    # Build dataframe. Each row contains the start and end frame of a shot.
    df_shots = pd.DataFrame(video["shots"], columns=["start", "end"])

    # Calculate the middle frame of a shot.
    df_shots["middle"] = df_shots["start"] + ((df_shots["end"] - df_shots["start"]) / 2).astype(int)

    # Calculate the duration of the shot in number of frames
    df_shots["duration"] = df_shots["end"] - df_shots["start"] + 1

    # Convert frames to seconds.
    # Time in seconds = frame number divided by frames per second ("frame_rate") of the respective video.
    for col in ["start", "end", "middle", "duration"]:
        df_shots[f"{col}_s"] = round(df_shots[col] / video["metadata"]["frame_rate"], 2)

    # Preview the first five detected shots.
    print(f"First five shots in {video_name}:")
    print(df_shots.head(5))
    print(DIVIDER)

    # Add the complete table to the corresponding video dictionary
    video.update({
        "df_shots" : df_shots
    })

First five shots in Der geteilte Himmel_Trailer_1964:
   start   end  middle  duration  start_s  end_s  middle_s  duration_s
0      0   179      89       180     0.00   7.16      3.56        7.20
1    180   447     313       268     7.20  17.88     12.52       10.72
2    448   788     618       341    17.92  31.52     24.72       13.64
3    789   971     880       183    31.56  38.84     35.20        7.32
4    972  1186    1079       215    38.88  47.44     43.16        8.60
────────────────────────────────────────────────────────────────────────────────────────────────────


### Berechnung der Analysemetadaten

Neben den Ergebnissen der Einstellungserkennung erzeugt das Notebook pro Video eine weitere Tabelle `metadata`, in der die Metadaten zur Analyse eingetragen werden.

Eine vollständige Übersicht der exportierten Daten bietet die Metadatentabelle im OER-Abschnitt <a href="https://quadriga-dk.github.io/Bewegtes-Bild-Fallstudie-4/annotation/einstellung_intro.html#filmname-shots-threshold-metadata-csv" class="external-link" target="_blank">zur Ausgabe des Notebooks</a>.

Diese Daten sind nützlich, um eine kompakte Übersicht über das Ergebnis zu erhalten oder um etwa die Auswirkungen unterschiedlicher `THRESHOLD`-Werte auf die Erkennungsergebnisse zu vergleichen. Außerdem werden diese Daten für die <a href="https://quadriga-dk.github.io/Bewegtes-Bild-Fallstudie-4/auswertung/visualisierung_einstellungen.html" class="external-link" target="_blank">Visualisierung der Einstellungsmetadaten im nächsten Kapitel</a> verwendet. 

In [ ]:
# Calculate the metadata for each previously registered video in sequence.
for video_name, video in videos.items():
    video["metadata"].update({
        # Record the threshold used to ensure reproducibility.
        "threshold"                 : THRESHOLD,

        # Number of shots detected by the model.
        "shot_count"                : len(video["df_shots"]),

        # Descriptive statistics for shot duration in seconds.
        "mean_duration_s"           : round(video["df_shots"]["duration_s"].mean(), 2),
        "median_duration_s"         : round(video["df_shots"]["duration_s"].median(), 2),
        "max_duration_s"            : round(video["df_shots"]["duration_s"].max(), 2),
        "min_duration_s"            : round(video["df_shots"]["duration_s"].min(), 2),

        # Duration of the source video and time required for automatic processing.
        "duration_hh_mm_ss"          : video["metadata"]["duration_hh_mm_ss"],
        "processing_time_hh_mm_ss"  : video["elapsed_time_hh_mm_ss"],

        # Range of probability values produced by the model.
        "min_prediction_score"      : round(float(video["prediction_scores"].min()),2),
        "max_prediction_score"      : round(float(video["prediction_scores"].max()),2),            
    })

    # Transform the metadata dictionary into a dataframe.
    video["metadata_df"] = pd.DataFrame.from_dict(
        video["metadata"], orient="index", columns=["Value"]
    )
    print(f"Shot-metadata for {video_name}:")
    # Display the last seven rows.
    display(video["metadata_df"].tail(7))
    print(DIVIDER)

Metadata for Der geteilte Himmel_Trailer_1964:


,Value
shot_count,23
mean_duration_s,9.09
median_duration_s,7.76
max_duration_s,22.04
min_duration_s,1.96
min_prediction_score,0.0
max_prediction_score,0.99


────────────────────────────────────────────────────────────────────────────────────────────────────


---
## Export der Ergebnisse

Die Analyseergebnisse werden als Nächstes in zwei Formaten gespeichert. 

`.csv`-Dateien eignen sich für Tabellenkalkulation, Statistikprogramme und weitere Python-Auswertungen, etwa zur Visualisierung. Zudem können sie auch  in *OpenRefine* eingelesen werden. Die Verwendung von *OpenRefine* wird in der <a href="https://quadriga-dk.github.io/Bewegtes-Bild-Fallstudie-2/bereinigung/openRefine/1_einf%C3%BChrung.html" class="external-link" target="_blank">OER zu Studentischen Filmen an der Filmuniversität</a> genauer erklärt, ebenso das <a href="https://quadriga-dk.github.io/Bewegtes-Bild-Fallstudie-2/bereinigung/openRefine/2_import.html#datenformate" class="external-link" target="_blank">csv-Dateiformat</a>.

Das `.eaf`-Format kann direkt in <a href="https://www.vian.app" class="external-link" target="_blank">*VIAN*</a> oder anderen Programmen zur Videoannotation geöffnet werden, um dort die Daten manuell zu überprüfen, zu bearbeiten und zu ergänzen.

Existiert bereits mindestens eine Datei dieses Satzes im `EXPORT_PATH`-Ordner, wird für alle Dateien ein Versionszusatz vergeben – `_v2`, `_v3` usw.

### Export der Tabellen als `.csv`-Dateien

Die Ergebnisse der Einstellungserkennung werden für jedes analysierte Video in zwei `.csv`-Dateien exportiert. Ihre Namen setzen sich aus dem Namen der Videodatei, `shots` und dem gewählten Schwellenwert (`THRESHOLD`) zusammen.

- Die Datei `filename_shots_THRESHOLD.csv` enthält die einzelnen erkannten Einstellungen. Jede Zeile dokumentiert Beginn, Ende, Mittelpunkt und Dauer einer Einstellung, jeweils in Frames und Sekunden.
- Die Datei `filename_shots_THRESHOLD_metadata.csv` enthält die Metadaten des Videos und der Einstellungserkennung.

Weitere Informationen finden Sie im OER-Abschnitt <a href="https://quadriga-dk.github.io/Bewegtes-Bild-Fallstudie-4/annotation/einstellung_intro.html#ausgabe" class="external-link" target="_blank">zur Ausgabe des Notebooks</a>.

Beide Dateien werden auf der obersten Ebene im zuvor festgelegten Exportordner (`EXPORT_PATH`) gespeichert.


In [10]:
# Select one version for all export files belonging to the same video.
# Find a free version for the complete group of CSV and EAF files.
def next_export_paths(paths):
    version = 1
    while True:
        # The first export has no suffix; later versions use _v2, _v3, etc.
        suffix = "" if version == 1 else f"_v{version}"
        candidates = {
            kind: path.with_name(f"{path.stem}{suffix}{path.suffix}")
            for kind, path in paths.items()
        }
        # Even one existing file reserves this version for the whole group.
        if not any(path.exists() for path in candidates.values()):
            return candidates
        version += 1

# Export the tables for each previously registered video.
export_directory = Path(EXPORT_PATH)
export_directory.mkdir(parents=True, exist_ok=True)

for video_name, video in videos.items():
    # The threshold in the filename distinguishes different model settings.
    stem = f"{video_name}_shots_{THRESHOLD}"

    # Reserve one shared version for both CSV files and the later EAF file.
    video["export_paths"] = next_export_paths({
        "shots": export_directory / f"{stem}.csv",
        "metadata": export_directory / f"{stem}_metadata.csv",
        "eaf": export_directory / f"{stem}.eaf",
    })
    paths = video["export_paths"]

    # mode="x" refuses to overwrite a CSV file that already exists.
    video["df_shots"].to_csv(paths["shots"], index=False, mode="x")

    # Store the metadata as key-value pairs without a column heading.
    video["metadata_df"].to_csv(paths["metadata"], header=False, mode="x")

    print(f"Successfully exported {paths['shots']} & {paths['metadata']}.")
    print(DIVIDER)


Successfully exported export_shots/Der geteilte Himmel_Trailer_1964_shots_0.5_v7.csv & export_shots/Der geteilte Himmel_Trailer_1964_shots_0.5_metadata_v7.csv.
────────────────────────────────────────────────────────────────────────────────────────────────────


### Export der Einstellungsannotationen als `.eaf`-Datei

Die Datei `filename_shots_THRESHOLD.eaf` enthält die erkannten Einstellungen als zeitliche Annotationen und wird ebenfalls direkt im Exportordner (`EXPORT_PATH`) gespeichert.

Für jedes Video wird eine Annotationsebene angelegt, die sich aus `shots` und dem gewählten Schwellenwert `THRESHOLD` zusammensetzt. Jede erkannte Einstellung erscheint dort als eigenes Zeitintervall. Die Intervalle enthalten anders als gewöhnlich keinen Text, sondern bilden eine zeitliche Segmentierung, die nach dem Import in VIAN dort auf ihre korrekte Erkennung hin überprüft und inhaltlich ergänzt werden kann.

In [11]:
# Export the .eaf annotations for each previously registered video in sequence.
for video_name, video in videos.items():
    # Reuse the version selected when exporting this video's CSV files.
    if "export_paths" not in video:
        raise RuntimeError("Run the CSV export cell before exporting EAF files.")
    eaf_document_path = video["export_paths"]["eaf"]

    # Prevent overwriting when the EAF export cell is run a second time.
    if eaf_document_path.exists():
        print(f"Skipping {eaf_document_path.name}: file already exists. Re-run the CSV export cell for a new version.")
        print(DIVIDER)
        continue

    frame_rate = float(video["metadata_df"].loc["frame_rate", "Value"])
  
    # Create a new, initially empty .eaf document.
    eaf_document = pympi.Eaf()
    # Add the annotation tier for the automatically detected shots.
    eaf_document.add_tier(f"shots_{THRESHOLD}")

    # Convert each table row into a time interval in milliseconds needed for the .eaf format.
    for _, row in video["df_shots"].iterrows():
        eaf_document.add_annotation(
            f"shots_{THRESHOLD}",
            round(row["start"] / frame_rate * 1000),
            round((row["end"] + 1) / frame_rate * 1000)
        )

    # Write the EAF using the version already assigned to the CSV files.
    eaf_document.to_file(eaf_document_path)

    print(f"Successfully exported {eaf_document_path}.")
    print(DIVIDER)

Successfully exported export_shots/Der geteilte Himmel_Trailer_1964_shots_0.5_v7.eaf.
────────────────────────────────────────────────────────────────────────────────────────────────────


### Herunterladen des *Google Colab* Export-Ordners

Dieser optionale Schritt ist nur für eine Ausführung in *Google Colab* vorgesehen. Der gesamte Exportordner in *Colab* wird zu einer `.zip`-Datei komprimiert und anschließend über die Browseroberfläche heruntergeladen. Der Download ist notwendig, da ansonsten die Erkennungsdaten nach Beenden der Laufzeit bzw. von *Colab* immer gelöscht werden, ebenso wie alle in Colab zur Analyse hochgeladenen Videodateien.

In einer lokalen *Jupyter Notebook*-Umgebung bleiben die Dateien im [anfangs](#festlegen-der-parameter) definierten Export-Ordner erhalten und können von dort entnommen werden. Es ist daher kein zusätzlicher Download-Schritt erforderlich.


In [12]:
try:
  # The following import works only within a Google Colab runtime.
  import google.colab
  from google.colab import files

  # Colab uses the `/content` directory by default.
  colab_export_directory = f"/content/{EXPORT_PATH}"
  zip_path = f"{EXPORT_PATH}.zip"
  # Package the entire results directory as a ZIP archive.
  !zip -r "{zip_path}" "{colab_export_directory}"

  # Start downloading the ZIP archive in the browser.
  files.download(f"/content/{EXPORT_PATH}.zip")
  print(f"Downloaded {EXPORT_PATH}.zip.")
except ImportError:
  # Skip this step when running locally.
  print("Download only works in Google Colab.")

Download only works in Google Colab.
